# Quantification of methylation data

## Description

This notebook implements two methods to quantify methylation data, using `sesame` and `minfi`. We recommend `sesame` over `minfi`.


|Procedure|`minfi`|`sesame`|
|--------------|-------|---------------|
|SNP/Cross reaction removal |  dropLociWithSnps + manual removal  | Q (qualityMask)  |
|sample quality|detectionP + mean |sesameQC_calcStats + "detection" + frac_dt |
|Bias correction|preprocessQuantile|D ( dyeBiasNL)|
|Probe quality|detectionP|"P (pOOBAH	Detection p-value masking using oob)" |
|Background substraction|NA|B (noob)|

## Input

1. `sample_sheet`: path to csv/tsv file that documenting all the meta-information of the bisulfite sequencing. The user need to manually ensure/rename the column names corresponding  to the first and second half of the idat file names are "Sentrix_ID" and "Sentrix_Position" 
2. [optional] `idat_folder`: path to the folder containing all the IDAT files to generate methylation data matrices from. Default is set to using the same folder where `sample_sheet` locates.
3. [optional] `cross_reactive_probes`: A list of CpG probes that are reported to [map to multiple regions in the genome.](https://academic.oup.com/nargab/article/2/4/lqaa105/6040968) 

## Output

For each input array, methylation is called with both SeSAMe and minfi and written under `--cwd`:

- `{sample}.sesame.beta.tsv` and `{sample}.sesame.M.tsv` — SeSAMe beta-values (proportion methylated) and M-values (logit-transformed) per probe; `{sample}.sesame.rds` stores the SeSAMe object and `{sample}.sample_qcs.sesame.tsv` the per-sample QC metrics.
- `{sample}.minfi.beta.tsv` and `{sample}.minfi.M.tsv` — the equivalent beta- and M-value matrices from minfi (with `{sample}.minfi.rds`).
- `{name}.bed.gz` (a pair for `beta` and `M`) and `{name}.gene_id.annot.tsv` — the final BED-format methylation matrices and the probe-to-gene annotation used as the molecular phenotype downstream.


## Steps




In [ ]:
sos run pipeline/methylation_calling.ipynb sesame \
    --sample-sheet data/MWE/MWE_Sample_sheet.csv \
    --container containers/methylation.sif

sos run pipeline/methylation_calling.ipynb sesame \
    --sample-sheet data/MWE/MWE_Sample_sheet_int.csv \
    --container containers/methylation.sif --sample_sheet_header_rows 0

In [ ]:
sos run pipeline/methylation_calling.ipynb minfi \
    --sample-sheet data/MWE/MWE_Sample_sheet.csv \
    --container containers/methylation.sif

## Command interface

In [1]:
sos run methylation_calling.ipynb -h

usage: sos run methylation_calling.ipynb
               [workflow_name | -t targets] [options] [workflow_options]
  workflow_name:        Single or combined workflows defined in this script
  targets:              One or more targets to generate
  options:              Single-hyphen sos parameters (see "sos run -h" for details)
  workflow_options:     Double-hyphen workflow-specific parameters

Workflows:
  sesame
  minfi

Global Workflow Options:
  --cwd output (as path)
                        The output directory for generated files.
  --sample-sheet VAL (as path, required)
                        The companion sample sheet csv file as outlined in the
                        input section.
  --idat-folder  path(f"{sample_sheet:d}")

                        Raw data folder
  --[no-]keep-only-cpg-probes (default to False)
                        Remove probes that are SNPs
  --job-size 1 (as int)
                        For cluster jobs, number commands to run per job
  --walltime 5h


## Global parameters

`keep_only_cpg_probes` option dictate whether only cpg probes should be kept:

- On an Illumina methylation bead chip, there are three types of probes,whose nature were indicated by their names.
       - cg: cpg probe;
       - rs: explict snp probe;
       - ch: non-CpG targeting probes; [reported to be more prone to cross-hybirdization](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC4909830/)
        
  Following the guideline of [Zhou W 2016](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC5389466/), by default we do not remove all the rs and ch probes. However, for research that are focusing on the CpG sites, like mQTL discovery, we should use `keep_only_cpg_probes` parameter to filter out other types of probes.
  

In [ ]:
[global]
# The output directory for generated files.
parameter: cwd = path("output")
# The companion sample sheet csv file as outlined in the input section.
parameter: sample_sheet = path
# Raw data folder
parameter: idat_folder = path(f"{sample_sheet:d}")
parameter: modular_script_dir = path('code/script')  # override with --modular-script-dir
# Remove probes that are SNPs
parameter: keep_only_cpg_probes = False
# For cluster jobs, number commands to run per job
parameter: job_size = 1
# Wall clock time expected
parameter: walltime = "5h"
# Memory expected
parameter: mem = "16G"

# Number of threads
parameter: numThreads = 8
# Software container option
parameter: container = ""
cwd = path(f'{cwd:a}')

## `Sesame` 

Getting the beta value from EPIC450 IDAT for 750 samples from 3000 wells take ~40 mins.

Based on [sesame documentation](https://www.bioconductor.org/packages/release/bioc/vignettes/sesame/inst/doc/sesame.html), the processing procedure suitable for human on EPIC 450 and 850 platform is "QCDPB"

The code for each processing procedure are as followed:
_____________

| Code | Name | Detail |
| -----|------|--------|
| Q | qualityMask | Mask probes of poor design | 
| C | inferInfiniumIChannel | Infer channel for Infinium-I probes | 
| D | dyeBiasNL | Dye bias correction (non-linear) | 
| P | pOOBAH | Detection p-value masking using oob | 
| B | noob | Background subtraction using oob | 



Other potential procedures are 


| Code | Name | Detail |
| -----|------|--------|
|0|resetMask|Reset mask to all FALSE |
|G|prefixMaskButCG|Mask all but cg- probes |
|H|prefixMaskButC|Mask all but cg- and ch-probes |
|E|dyeBiasL|Dye bias correction (linear) |
|I|detectionIB|Mask detection by intermediate beta values |
 

M value is calculated as M = log2(beta/(1-beta))
The way we handle beta == 0 or beta == 1 is by replacing them with the next min/max value among the beta matrix, which is based on [here](https://github.com/xuz1/ENmix/blob/master/R/utils.R) 



In [ ]:
[sesame_1]
# threshold to filter out samples based on frac_dt (Percentage of probe Detection Success)  percentage
parameter: samples_frac_dt_cutoff = 0.8
# The header rows in the sample sheet csv. Use 0 for no headers. Typically it should be 7.
parameter: sample_sheet_header_rows = float
# The number of cores to use. If 0, determined by BiocParallel::multicoreWorkers().
parameter: n_cores = 1

input: sample_sheet
output: f'{cwd}/{_input:bn}.sesame.rds',f'{cwd}/{_input:bn}.sesame.beta.tsv',f'{cwd}/{_input:bn}.sesame.M.tsv',f'{cwd}/{_input:bn}.sample_qcs.sesame.tsv'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads
bash: expand= "${ }", stderr = f'{_output[0]:n}.stderr', stdout = f'{_output[0]:n}.stdout', container=container
    Rscript ${modular_script_dir}/molecular_phenotypes/calling/methylation_calling.R --step sesame \
        --sample-sheet "${_input}" \
        --idat-folder "${idat_folder}" \
        --sample-sheet-header-rows ${sample_sheet_header_rows} \
        --samples-frac-dt-cutoff ${samples_frac_dt_cutoff} \
        --n-cores ${n_cores} \
        ${'--keep-only-cpg-probes' if keep_only_cpg_probes else ''} \
        --output-rds "${_output[0]}" \
        --output-beta "${_output[1]}" \
        --output-m "${_output[2]}" \
        --output-qcs "${_output[3]}"

## `minfi`

By default, for Infinium MethylationEPIC the data will be annotated based on hg38 using [this annotation](https://github.com/achilleasNP/IlluminaHumanMethylationEPICanno.ilm10b5.hg38), alternatively user can set the `--hg-build` parameter back to 19 to use the [hg19 annotation](https://bioconductor.org/packages/release/data/annotation/html/IlluminaHumanMethylationEPICanno.ilm10b4.hg19.html).

For 450K data however, only hg19 annotation is availble, which is what we would use would use for minfi to work. However, we will reannotate everything to hg38 anyways in the next step. 



1. All the IDAT file in the specified folder and sub-folder will be loaded for samples in input sample CSV file
2. The methylation data samples will first be filtered based on [bisulphite conversation rate](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC4527772/). This operation is done using the [bscon function from watermelon package](http://www.bioconductor.org/packages/release/bioc/vignettes/wateRmelon/inst/doc/wateRmelon.html#introduction) 
3. samples will then be filtered based on a [detection pvalue](https://www.rdocumentation.org/packages/minfi/versions/1.18.4/topics/detectionP), which indicates the quality of the signal at each genomics position
4. [Stratified Quantile Normalization](https://rdrr.io/bioc/minfi/man/preprocessQuantile.html) will then be applied.
5. features will be filtered if they are on sex chr, known to be [cross-reactive,maping to multiple regions in the genome.](https://academic.oup.com/nargab/article/2/4/lqaa105/6040968), overlapping with snps, or having too low a detection P. The list of cross-reactive probe can be found as `/opt/cross_reactive_probe_Hop2020.txt` in our docker and [here](https://raw.githubusercontent.com/hsun3163/xqtl-protocol/main/data/cross_reactive_probe_Hop2020.txt).
6. Beta and M value will for all the probes/samples will then each be saved to a indexed bed.gz file.

[As documented here](https://github.com/statfungen/xqtl-protocol/issues/312) when the batch of IDAT data are different, there will be a problem reading the IDAT file without specifing the force = TRUE option in the `read.metharray.exp(targets = targets,force = TRUE)`

In [ ]:
[minfi_1]
# threshold to filter out samples based on detection P value
parameter: samples_pval_cutoff = 0.05
# threshold to filter out probes based on detection P value
parameter: probe_pval_cutoff = 0.01
# Cross-reactive probe list (data/cross_reactive_probe_Hop2020.txt); "." to skip
parameter: cross_reactive_probes = path("data/cross_reactive_probe_Hop2020.txt")
# 38 (hg38) or 19 (hg19) for epic data, by default 38. Noted for 450K data only GRCh37 is availble
parameter: hg_build = 38
input: sample_sheet
output: f'{cwd}/{_input:bn}.minfi.rds',f'{cwd}/{_input:bn}.minfi.beta.tsv',f'{cwd}/{_input:bn}.minfi.M.tsv'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads
bash: expand= "${ }", stderr = f'{_output[0]:n}.stderr', stdout = f'{_output[0]:n}.stdout', container=container
    Rscript ${modular_script_dir}/molecular_phenotypes/calling/methylation_calling.R --step minfi \
        --sample-sheet "${_input}" \
        --samples-pval-cutoff ${samples_pval_cutoff} \
        --probe-pval-cutoff ${probe_pval_cutoff} \
        --cross-reactive-probes "${cross_reactive_probes}" \
        --hg-build ${hg_build} \
        ${'--keep-only-cpg-probes' if keep_only_cpg_probes else ''} \
        --output-rds "${_output[0]}" \
        --output-beta "${_output[1]}" \
        --output-m "${_output[2]}"

## Annotate probes

The probes are annotated via `sesameData` package and formatted as bgzipped bed files, regardless of method used to process the IDAT.

In [ ]:
[*_2]
output: f'{_input[1]:n}.bed.gz', f'{_input[2]:n}.bed.gz', f'{_input[0]:n}.gene_id.annot.tsv'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads
bash: expand= "${ }", stderr = f'{_output[0]:n}.stderr', stdout = f'{_output[0]:n}.stdout', container=container
    Rscript ${modular_script_dir}/molecular_phenotypes/calling/methylation_calling.R --step annotate \
        --input-beta "${_input[1]}" \
        --input-m "${_input[2]}" \
        --output-beta-bed "${_output[0]}" \
        --output-m-bed "${_output[1]}" \
        --output-annot "${_output[2]}"

## Anticipated Results

The pipeline produces a methylation beta-value matrix (one row per CpG probe, one column per sample) with SNP-overlapping probes removed and missing values imputed, ready for downstream mQTL analysis.
